# Python-часть тестового задания
### Библиотека: **polars**



## 0. Установка зависимостей

In [ ]:
# Устанавливаем нужные библиотеки (выполнить один раз)
!pip install polars holidays -q

---
## Загрузка CSV-файлов в Google Colab
Загрузите **orders.csv** и **product_info.csv**.


In [2]:
from google.colab import files

# Открываем диалог загрузки файлов
uploaded = files.upload()

# Убедимся, что оба файла загружены
print("Загруженные файлы:", list(uploaded.keys()))

Saving orders.csv to orders.csv
Saving product_info.csv to product_info.csv
Загруженные файлы: ['orders.csv', 'product_info.csv']


---
## Задание 1. Загрузка и первичный просмотр
Загружаем оба CSV в DataFrame polars и смотрим первые 5 строк каждого.  


In [3]:
import polars as pl

# Загружаем orders.csv; try_parse_dates автоматически парсит даты
orders = pl.read_csv("orders.csv", try_parse_dates=True)

# Загружаем product_info.csv
products = pl.read_csv("product_info.csv")

print("=== orders — первые 5 строк ===")
print(orders.head())

print(f"\nРазмер: {orders.shape[0]} строк, {orders.shape[1]} колонок")
print("Типы данных:")
print(orders.dtypes)

print("\n=== product_info — первые 5 строк ===")
print(products.head())

print(f"\nРазмер: {products.shape[0]} строк, {products.shape[1]} колонок")
print("Типы данных:")
print(products.dtypes)

=== orders — первые 5 строк ===
shape: (5, 7)
┌──────────┬────────────┬───────────────┬────────────┬──────────┬───────┬─────────────┐
│ order_id ┆ order_date ┆ delivery_date ┆ product_id ┆ quantity ┆ price ┆ region      │
│ ---      ┆ ---        ┆ ---           ┆ ---        ┆ ---      ┆ ---   ┆ ---         │
│ i64      ┆ date       ┆ date          ┆ i64        ┆ i64      ┆ i64   ┆ str         │
╞══════════╪════════════╪═══════════════╪════════════╪══════════╪═══════╪═════════════╡
│ 1        ┆ 2023-01-10 ┆ 2023-01-15    ┆ 1          ┆ 2        ┆ 50000 ┆ Центральный │
│ 2        ┆ 2023-01-12 ┆ 2023-01-18    ┆ 2          ┆ 5        ┆ 30000 ┆ Восточный   │
│ 3        ┆ 2023-01-14 ┆ 2023-01-20    ┆ 3          ┆ 10       ┆ 500   ┆ Западный    │
│ 4        ┆ 2023-01-15 ┆ 2023-01-19    ┆ 4          ┆ 15       ┆ 2000  ┆ Восточный   │
│ 5        ┆ 2023-01-18 ┆ 2023-01-25    ┆ 5          ┆ 3        ┆ 8000  ┆ Центральный │
└──────────┴────────────┴───────────────┴────────────┴──────────┴───────┴─

---
## Задание 2. Фильтрация
Оставляем только заказы, где `region = "Восточный"` и `quantity > 10`.


In [4]:
# Фильтруем: регион "Восточный" И количество > 10
east_large = orders.filter(
    (pl.col("region") == "Восточный") & (pl.col("quantity") > 10)
)

print(f"Количество заказов (Восточный, quantity > 10): {len(east_large)}")
print()
print("Первые 5 строк:")
print(east_large.head())

Количество заказов (Восточный, quantity > 10): 17

Первые 5 строк:
shape: (5, 7)
┌──────────┬────────────┬───────────────┬────────────┬──────────┬───────┬───────────┐
│ order_id ┆ order_date ┆ delivery_date ┆ product_id ┆ quantity ┆ price ┆ region    │
│ ---      ┆ ---        ┆ ---           ┆ ---        ┆ ---      ┆ ---   ┆ ---       │
│ i64      ┆ date       ┆ date          ┆ i64        ┆ i64      ┆ i64   ┆ str       │
╞══════════╪════════════╪═══════════════╪════════════╪══════════╪═══════╪═══════════╡
│ 4        ┆ 2023-01-15 ┆ 2023-01-19    ┆ 4          ┆ 15       ┆ 2000  ┆ Восточный │
│ 6        ┆ 2023-01-20 ┆ 2023-01-24    ┆ 6          ┆ 100      ┆ 50    ┆ Восточный │
│ 25       ┆ 2023-02-09 ┆ 2023-02-17    ┆ 15         ┆ 20       ┆ 150   ┆ Восточный │
│ 42       ┆ 2023-02-26 ┆ 2023-03-06    ┆ 12         ┆ 15       ┆ 2800  ┆ Восточный │
│ 62       ┆ 2023-03-18 ┆ 2023-03-28    ┆ 12         ┆ 12       ┆ 2900  ┆ Восточный │
└──────────┴────────────┴───────────────┴────────────┴─────

---
## Задание 3. Объединение таблиц + вычисляемый столбец
Соединяем `orders` и `product_info` по ключу `product_id` (LEFT JOIN).  
Добавляем столбец `total_cost = quantity × price`.


In [5]:
# LEFT JOIN: все заказы + информация о товаре
merged = orders.join(products, on="product_id", how="left")

# Добавляем вычисляемый столбец total_cost
merged = merged.with_columns(
    (pl.col("quantity") * pl.col("price")).alias("total_cost")
)

print(f"Строк в объединённой таблице: {len(merged)}")
print(f"Колонки: {merged.columns}")
print()
print("Первые 5 строк:")
print(merged.head())

Строк в объединённой таблице: 200
Колонки: ['order_id', 'order_date', 'delivery_date', 'product_id', 'quantity', 'price', 'region', 'product_name', 'category', 'supplier', 'base_stock', 'total_cost']

Первые 5 строк:
shape: (5, 12)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ order_id ┆ order_dat ┆ delivery_ ┆ product_i ┆ … ┆ category  ┆ supplier  ┆ base_stoc ┆ total_cos │
│ ---      ┆ e         ┆ date      ┆ d         ┆   ┆ ---       ┆ ---       ┆ k         ┆ t         │
│ i64      ┆ ---       ┆ ---       ┆ ---       ┆   ┆ str       ┆ str       ┆ ---       ┆ ---       │
│          ┆ date      ┆ date      ┆ i64       ┆   ┆           ┆           ┆ i64       ┆ i64       │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1        ┆ 2023-01-1 ┆ 2023-01-1 ┆ 1         ┆ … ┆ Электрони ┆ Поставщик ┆ 120       ┆ 100000    │
│          ┆ 0         ┆ 5         ┆           ┆   ┆ ка      

---
## Задание 4. Рабочие дни между датами
Для каждого заказа считаем количество рабочих дней между `order_date` и `delivery_date`.

Используем:
- `numpy.busday_count` — считает рабочие дни, принимает список праздников
- `holidays.Russia` — официальные праздники РФ
- `map_elements` — построчное применение функции в polars (аналог `apply` в pandas)

> ⚠️ `map_elements` медленнее, чем встроенные выражения polars, но нужен  
> для логики с праздниками, которую нельзя выразить через `pl.col(...)`.


In [6]:
import holidays
import numpy as np

# Собираем праздники РФ для всех лет из датасета
years = merged["order_date"].dt.year().unique().to_list()
ru_holidays = holidays.Russia(years=years)

def count_business_days(row: dict) -> int:
    """
    Считает рабочие дни между двумя датами с учётом праздников РФ.
    Принимает dict с ключами 'order_date' и 'delivery_date'.
    """
    start = row["order_date"]
    end   = row["delivery_date"]

    if start is None or end is None:
        return None

    # Праздники только в диапазоне между датами заказа и доставки
    holiday_dates = [
        d for d in ru_holidays.keys()
        if start <= d <= end
    ]

    return int(np.busday_count(start, end, holidays=holiday_dates))

# map_elements применяет функцию к каждой строке struct из двух колонок
merged = merged.with_columns(
    pl.struct(["order_date", "delivery_date"])
    .map_elements(count_business_days, return_dtype=pl.Int64)
    .alias("business_days")
)

print("Столбец business_days добавлен. Проверка первых 10 строк:")
print(
    merged.select(["order_id", "order_date", "delivery_date", "business_days"])
    .head(10)
)

print(f"\nМин. рабочих дней: {merged['business_days'].min()}")
print(f"Макс. рабочих дней: {merged['business_days'].max()}")
print(f"Среднее рабочих дней: {merged['business_days'].mean():.2f}")

Столбец business_days добавлен. Проверка первых 10 строк:
shape: (10, 4)
┌──────────┬────────────┬───────────────┬───────────────┐
│ order_id ┆ order_date ┆ delivery_date ┆ business_days │
│ ---      ┆ ---        ┆ ---           ┆ ---           │
│ i64      ┆ date       ┆ date          ┆ i64           │
╞══════════╪════════════╪═══════════════╪═══════════════╡
│ 1        ┆ 2023-01-10 ┆ 2023-01-15    ┆ 4             │
│ 2        ┆ 2023-01-12 ┆ 2023-01-18    ┆ 4             │
│ 3        ┆ 2023-01-14 ┆ 2023-01-20    ┆ 4             │
│ 4        ┆ 2023-01-15 ┆ 2023-01-19    ┆ 3             │
│ 5        ┆ 2023-01-18 ┆ 2023-01-25    ┆ 5             │
│ 6        ┆ 2023-01-20 ┆ 2023-01-24    ┆ 2             │
│ 7        ┆ 2023-01-21 ┆ 2023-01-27    ┆ 4             │
│ 8        ┆ 2023-01-22 ┆ 2023-01-29    ┆ 5             │
│ 9        ┆ 2023-01-23 ┆ 2023-01-26    ┆ 3             │
│ 10       ┆ 2023-01-24 ┆ 2023-01-30    ┆ 4             │
└──────────┴────────────┴───────────────┴───────────────┘

---
## Задание 5. Агрегация по регионам
Группируем по `region` и считаем три метрики.


In [ ]:
region_stats = (
    merged
    .group_by("region")
    .agg([
        pl.col("business_days").mean().round(2).alias("avg_business_days_by_region"),
        pl.col("total_cost").sum().alias("total_sales_by_region"),
        pl.col("order_id").count().alias("order_count_by_region"),
    ])
    .sort("avg_business_days_by_region", descending=True)
)

print("Статистика по регионам (сортировка по среднему числу рабочих дней):")
print(region_stats)

Статистика по регионам (сортировка по среднему числу рабочих дней):
shape: (5, 4)
┌─────────────┬─────────────────────────────┬───────────────────────┬───────────────────────┐
│ region      ┆ avg_business_days_by_region ┆ total_sales_by_region ┆ order_count_by_region │
│ ---         ┆ ---                         ┆ ---                   ┆ ---                   │
│ str         ┆ f64                         ┆ i64                   ┆ u32                   │
╞═════════════╪═════════════════════════════╪═══════════════════════╪═══════════════════════╡
│ Южный       ┆ 6.18                        ┆ 3303480               ┆ 38                    │
│ Северный    ┆ 6.11                        ┆ 794176                ┆ 38                    │
│ Центральный ┆ 5.98                        ┆ 6860550               ┆ 41                    │
│ Западный    ┆ 5.95                        ┆ 6097345               ┆ 40                    │
│ Восточный   ┆ 5.91                        ┆ 5165800               ┆ 43

---
## Задание 6. Итоговый вывод результатов
Выводим обе таблицы в читаемом виде.


In [ ]:
# ── Итоговая таблица по регионам ──────────────────────────────────────────
print("=" * 60)
print("  СТАТИСТИКА ПО РЕГИОНАМ")
print("=" * 60)

# Переименовываем колонки для красивого вывода
print(
    region_stats.rename({
        "region":                       "Регион",
        "avg_business_days_by_region":  "Ср. рабочих дней",
        "total_sales_by_region":        "Сумма продаж (₽)",
        "order_count_by_region":        "Кол-во заказов",
    })
)

# ── Объединённая таблица (сводка) ─────────────────────────────────────────
print()
print("=" * 60)
print("  ОБЪЕДИНЁННАЯ ТАБЛИЦА — первые 10 строк")
print("=" * 60)

print(
    merged.select([
        "order_id", "order_date", "delivery_date",
        "product_name", "category", "region",
        "quantity", "price", "total_cost", "business_days"
    ])
    .head(10)
)

# ── Общие итоги ───────────────────────────────────────────────────────────
print()
print("=" * 60)
print("  ОБЩИЕ ИТОГИ")
print("=" * 60)
print(f"  Всего заказов:           {len(merged)}")
print(f"  Общая выручка:           {merged['total_cost'].sum():,.0f} ₽")
print(f"  Средний чек:             {merged['total_cost'].mean():,.0f} ₽")
print(f"  Среднее рабочих дней:    {merged['business_days'].mean():.2f}")

  СТАТИСТИКА ПО РЕГИОНАМ
shape: (5, 4)
┌─────────────┬──────────────────┬──────────────────┬────────────────┐
│ Регион      ┆ Ср. рабочих дней ┆ Сумма продаж (₽) ┆ Кол-во заказов │
│ ---         ┆ ---              ┆ ---              ┆ ---            │
│ str         ┆ f64              ┆ i64              ┆ u32            │
╞═════════════╪══════════════════╪══════════════════╪════════════════╡
│ Южный       ┆ 6.18             ┆ 3303480          ┆ 38             │
│ Северный    ┆ 6.11             ┆ 794176           ┆ 38             │
│ Центральный ┆ 5.98             ┆ 6860550          ┆ 41             │
│ Западный    ┆ 5.95             ┆ 6097345          ┆ 40             │
│ Восточный   ┆ 5.91             ┆ 5165800          ┆ 43             │
└─────────────┴──────────────────┴──────────────────┴────────────────┘

  ОБЪЕДИНЁННАЯ ТАБЛИЦА — первые 10 строк
shape: (10, 10)
┌──────────┬────────────┬────────────┬────────────┬───┬──────────┬───────┬────────────┬────────────┐
│ order_id ┆ order_da